In [ ]:
# Source - https://stackoverflow.com/a/5399339
# Posted by pv., modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-26, License - CC BY-SA 4.0

%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
import yaml
import pickle
from pathlib import Path
from typing import Any, Dict

# Disable XLA preallocation to avoid hogging GPU memory
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm
from absl import flags

import jax
import jax.numpy as jnp
from jax import config as jax_config
import optax
import haiku as hk
from flax.training.early_stopping import EarlyStopping
import wandb

import jax_cosmo as jc
from jaxpm.nn import CNN, NeuralSplineFourierFilter
from jaxpm.painting import compensate_cic
from jaxpm.utils import power_spectrum
from jaxpm.pm import get_delta

In [ ]:
import jax, jax.numpy as jnp
import numpy as np

# Debug: desactiva JIT temporalmente
jax.config.update("jax_disable_jit", True)

# (Opcional) Asegura float32 por default en tu notebook (no siempre necesario)
jax.config.update("jax_enable_x64", False)

print("JIT disabled =", jax.config.jax_disable_jit)
print("x64 enabled =", jax.config.jax_enable_x64)

In [ ]:
from pm2nbody.read_data import load_datasets
from pm2nbody.loss import (
    get_frozen_potential_loss,
    get_potential_loss,
    get_position_loss,
    get_mse_pos,
)

In [ ]:

# --- Global Configurations ---
jax_config.update("jax_enable_x64", False)

mpl.rcParams.update(
    {
        "text.usetex": False,
        "font.family": "serif",
    }
)
mpl.rcParams["figure.dpi"] = 300
plt.style.use("default")

DEFAULT_DATA_DIR = Path("/cosmos_storage/home/diegovillalba/JaxPM/data/")
DEFAULT_MODEL_DIR = Path("/cosmos_storage/home/diegovillalba/JaxPM/models/")



In [ ]:
# FLAGS = flags.FLAGS
# FLAGS(sys.argv)
cfg = {
        "data": {
            "mesh_lr": 128,
            "mesh_hr": 256,
            "n_train_sims": 1,
            "n_val_sims": 1,
            "n_test_sims": 1,
            "snapshots": None,
            "box_size": 256.0,
            "n_snapshots": 50,
            "n_particles": 128,
        },
        "correction_model": {
            "type": "cnn",
            "channels_hidden_dim": 16,
            "n_convolutions": 3,
            "n_fully_connected": 2,
            "input_dim": 1,
            "kernel_size": 3,
            "pad_periodic": True,
            "embed_globals": False,
            "n_globals_embedding": 1,
            "globals_embedding_dim": 64,
            "global_conditioning": "add",
            "use_attention_interpolation": False,
            "add_particle_velocities": True,
            "n_knots": 16,
            "latent_size": 64,
        },
        "training": {
            "seed": 0,
            "n_steps": 100,
            "batch_size": 1,
            "patience": 20,
            "checkpoint_every": 5,
            "sample_snapshots": True,
            "loss": "mse_positions",
            "weight_snapshots": True,
            "lambda_pos": 1.0,
            "lambda_velocity": 1.0,
            "lambda_density": 0.0,
            "lambda_pk": 0.0,
            "lambda_cross_corr": 0.0,
            "log_pos": False,
            "fractional_mse": False,
            "weight_decay": 1e-4,
            "max_idx": 49,
            "schedule": {
                "type": "cosine",
                "initial_lr": 0.0,
                "peak_value": 3e-4,
                "warmup_steps": 50,
                "n_steps": 20_000,
                "factor": 0.5,
                "patience": 5,
                "min_lr": 1e-6,
            },
        },
        "wandb": {
            "project": "pm2nbody",
        },
    }


In [ ]:
from types import SimpleNamespace

def dict_to_namespace(d):
    if isinstance(d, dict):
        return SimpleNamespace(**{
            k: dict_to_namespace(v) for k, v in d.items()
        })
    return d

config = dict_to_namespace(cfg)

In [ ]:
config.correction_model.type

In [ ]:
import sys
sys.path.append("/cosmos_storage/home/diegovillalba/JaxPM/pm2nbody/")
from train_refactored import build_network
from train_refactored import build_dataloader
from train_refactored import initialize_network
from train_refactored import build_loss_fn
from train_refactored import build_schedule
from train_refactored import build_optimizer
from train_refactored import print_initial_lr_loss
from train_refactored import plot_eval

In [ ]:
neural_net = build_network(config.correction_model)

In [ ]:
cosmology, scale_factors, train_data, val_data, test_data = build_dataloader(
        config.data, data_dir=DEFAULT_DATA_DIR
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_lr_vs_hr(
    batch,
    *,
    max_idx=-1,
    mesh_plot=None,
    box_size=256.0,
    slab_thickness=8,
    plot_log=True,
    assume_mesh_units=True,
    show_pk=True,
):
    """
    Plot only LR vs HR from a batch (no model prediction needed).

    batch: {"lr": ResolutionData, "hr": ResolutionData}
    assume_mesh_units:
      - True: positions are already in [0, mesh) units
      - False: positions are in [0, 1) box units and will be scaled by mesh_plot
    """

    if mesh_plot is None:
        mesh_plot = int(batch["lr"].mesh)

    def to_mesh(pos):
        pos = np.asarray(pos)
        if assume_mesh_units:
            return pos
        return pos * mesh_plot

    # positions at snapshot max_idx
    pos_lr = to_mesh(batch["lr"].positions[max_idx])
    pos_hr = to_mesh(batch["hr"].positions[max_idx])

    # wrap to [0, mesh_plot)
    pos_lr = np.mod(pos_lr, mesh_plot)
    pos_hr = np.mod(pos_hr, mesh_plot)

    # deposit delta fields
    delta_lr = get_delta(pos_lr, (mesh_plot, mesh_plot, mesh_plot))
    delta_hr = get_delta(pos_hr, (mesh_plot, mesh_plot, mesh_plot))

    # slab around center
    z0 = mesh_plot // 2
    h = max(1, slab_thickness // 2)
    sl = slice(max(z0 - h, 0), min(z0 + h, mesh_plot))

    proj_lr = delta_lr[:, :, sl].sum(axis=-1)
    proj_hr = delta_hr[:, :, sl].sum(axis=-1)

    if plot_log:
        eps = 1e-4
        proj_lr = np.log10(np.clip(1.0 + proj_lr, eps, None))
        proj_hr = np.log10(np.clip(1.0 + proj_hr, eps, None))
        title_suffix = " (Log)"
    else:
        title_suffix = ""

    # ---- Figure: LR vs HR ----
    fig, ax = plt.subplots(ncols=2, figsize=(10, 4))
    cmap = "cividis"

    ax[0].imshow(proj_lr, cmap=cmap)
    ax[0].set_title(f"LR{title_suffix}")
    ax[1].imshow(proj_hr, cmap=cmap)
    ax[1].set_title(f"HR{title_suffix}")


    plt.show()

    # ---- Optional: Power spectrum ratio ----
    if show_pk:
        def get_pk(delta):
            delta_np = np.asarray(delta)
            delta_np = compensate_cic(delta_np)
            k, pk = power_spectrum(
                delta_np,
                boxsize=np.array([box_size] * 3),
                kmin=np.pi / box_size,
                dk=2 * np.pi / box_size,
            )
            return np.asarray(k), np.asarray(pk)

        k, pk_hr = get_pk(delta_hr)
        _, pk_lr = get_pk(delta_lr)

        valid = np.isfinite(k) & np.isfinite(pk_hr) & (pk_hr > 0) & np.isfinite(pk_lr)
        k = k[valid]
        pk_hr = pk_hr[valid]
        pk_lr = pk_lr[valid]

        plt.figure(figsize=(7, 5))
        plt.axhline(1.0, linestyle="dashed")
        if k.size > 0:
            plt.semilogx(k, pk_lr / pk_hr, label="LR / HR")
            plt.legend()
        else:
            plt.text(0.5, 0.5, "No valid P(k) bins", ha="center", va="center", transform=plt.gca().transAxes)

        plt.xlabel(r"$k$ [$h \ \mathrm{Mpc}^{-1}$]")
        plt.ylabel(r"$P(k)/P_\mathrm{HR}(k)$")
        plt.title("Power spectrum ratio")
        plt.show()

In [ ]:
plot_lr_vs_hr(
    test_data[0],
    max_idx=-1,
    slab_thickness=64,
    plot_log=True,
    assume_mesh_units=True,  
    show_pk=True,
)

In [ ]:
loss_fn = build_loss_fn(
        config.training, neural_net, cosmology,
    correction_type=config.correction_model.type,
    mesh_lr=train_data[0]["lr"].mesh,
)

print("scale_factors:", scale_factors.shape, scale_factors.dtype)
print("mesh_lr:", train_data[0]["lr"].mesh, "mesh_hr:", train_data[0]["hr"].mesh)

In [ ]:
batch_cpu = next(train_data.iterator)        # objetos del dataset (CPU)
dev = jax.devices()[0]
batch = train_data.move_to_device(batch_cpu, device=dev)  # batch para correr (GPU/CPU)

lr = batch["lr"]
hr = batch["hr"]

print("Devices:", lr.positions.device, hr.positions.device)
print("Shapes:",
      "lr.pos", lr.positions.shape,
      "lr.vel", lr.velocities.shape,
      "hr.pos", hr.positions.shape,
      "hr.vel", hr.velocities.shape)
print("mesh_lr in batch:", lr.mesh, "mesh_hr in batch:", hr.mesh)

In [ ]:
mesh_lr = lr.mesh
mesh_hr = hr.mesh

# max típico de pos
mx_lr = float(jnp.max(lr.positions[49]))
mx_hr = float(jnp.max(hr.positions[49]))

print("max lr.pos[t0] =", mx_lr)
print("max hr.pos[t0] =", mx_hr)
print("mesh_lr =", mesh_lr, "mesh_hr =", mesh_hr)

# Heurística:
# - si mx_lr ~ mesh_lr y mx_hr ~ mesh_lr => consistente (ambos mesh_lr units)
# - si mx_lr ~ 1 y mx_hr ~ 1 => consistente (ambos unit box) PERO tu loss debe esperar eso
# - si mx_lr ~ mesh_lr y mx_hr ~ mesh_hr => inconsistente y rompe % n_mesh

In [ ]:
params = initialize_network(
        train_data[0], neural_net=neural_net, model_type=config.correction_model.type
    )

In [ ]:
schedule = build_schedule(config.training.schedule)
optimizer, opt_state = build_optimizer(
        config.training, params=params, schedule=schedule
    )

In [ ]:
print_initial_lr_loss(val_data)

early_stop = EarlyStopping(patience=config.training.patience)
best_params = params
best_loss = float("inf")

In [ ]:
out = loss_fn(params, batch, scale_factors)
print("loss_fn output type:", type(out))
if isinstance(out, tuple):
    loss_val, aux = out
else:
    loss_val, aux = out, None
print("loss =", float(loss_val))
print("aux is None?", aux is None)

In [ ]:
import jax.numpy as jnp

mesh_lr = batch["lr"].mesh
pos_lr0 = batch["lr"].positions[0]
pos_hr0 = batch["hr"].positions[0]
pos_pm0 = aux[0]  # si aux es pos_pm con shape (T, Np, 3)

def quick_stats(name, x):
    return (name,
            float(jnp.min(x)), float(jnp.max(x)),
            float(jnp.mean(x)), float(jnp.std(x)),
            bool(jnp.isnan(x).any()), bool(jnp.isinf(x).any()))

print("mesh_lr =", mesh_lr)
print(quick_stats("pos_lr[t0]", pos_lr0))
print(quick_stats("pos_hr[t0]", pos_hr0))
print(quick_stats("pos_pm[t0]", pos_pm0))

m = jnp.asarray(mesh_lr, dtype=pos_pm0.dtype)

# MSE en mesh units (como lo estás viendo ahora, probablemente)
mse_mesh = jnp.mean((pos_pm0 - pos_hr0) ** 2)

# MSE en box units
mse_box = jnp.mean(((pos_pm0/m) - (pos_hr0/m)) ** 2)

print("MSE mesh units =", float(mse_mesh))
print("MSE box units  =", float(mse_box))
print("ratio (mesh/box) ~", float(mse_mesh / (mse_box + 1e-30)))
print("mesh_lr^2 =", float(m*m))

T = aux.shape[0]
# revisa último snapshot (donde suele reventar)
pos_pml = aux[-1]
print(quick_stats("pos_pm[last]", pos_pml))

# ¿Cuánto se diferencia HR cuando haces modulo mesh_lr?
pos_hr_mod = jnp.mod(batch["hr"].positions[-1], mesh_lr)
delta_mod = jnp.mean(jnp.abs(pos_hr_mod - batch["hr"].positions[-1]))
print("mean |mod(hr,pos)-hr| =", float(delta_mod))

In [ ]:
# ==========================================
# CELL 1: SETUP & EXTRACCIÓN DEL PRIMER LOTE
# ==========================================

# 1. Definimos el wrapper para que JAX pueda derivar respecto a 'params'
def train_loss_fn(p, batch, sf):
    out = loss_fn(params=p, dataset=batch, scale_factors=sf)
    return out[0] if isinstance(out, tuple) else out

# 2. Simulamos la selección dinámica del snapshot
# rng = jax.random.PRNGKey(42) # Semilla fija para debug
# if config.training.sample_snapshots:
#     rng, _ = jax.random.split(rng)
#     # max_idx = jax.random.randint(
#     #     rng, minval=10, maxval=len(scale_factors), shape=(1,)
#     # )[0]
#     max_idx = jnp.asarray(len(scale_factors)-1, dtype=jnp.int32)
# else:
#     max_idx = None

max_idx = jnp.asarray(len(scale_factors)-1, dtype=jnp.int32)
# 3. Extraemos UN SOLO batch manualmente
batch = next(train_data.iterator)
batch = train_data.move_to_device(batch, device=jax.devices()[0])

print(f"Max Index seleccionado: {max_idx}")
print(f"Shape de Posiciones LR: {batch['lr'].positions.shape}")
print(f"Shape de Posiciones HR (Target): {batch['hr'].positions.shape}")

In [ ]:
import subprocess

def gpu_mem_mib():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,nounits,noheader"]
    ).decode().strip().splitlines()
    # si hay varias GPUs, toma la 0
    used, total = out[0].split(",")
    return int(used), int(total)

def print_gpu_mem(tag=""):
    used, total = gpu_mem_mib()
    print(f"[GPU mem] {tag}: {used} MiB / {total} MiB")
import numpy as np
import jax
import jax.numpy as jnp
import jax.tree_util as tree

def pytree_nbytes(pytree):
    leaves = tree.tree_leaves(pytree)
    n = 0
    for x in leaves:
        # JAX array / numpy array / python scalar
        if hasattr(x, "dtype") and hasattr(x, "shape"):
            n += np.prod(x.shape) * np.dtype(x.dtype).itemsize
    return int(n)

def pretty_bytes(n):
    for unit in ["B","KiB","MiB","GiB","TiB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PiB"

def print_pytree_bytes(name, pytree):
    print(f"[{name}] approx bytes = {pretty_bytes(pytree_nbytes(pytree))}")

In [ ]:
import numpy as np

def arr_bytes(x):
    return int(np.prod(x.shape)) * np.dtype(x.dtype).itemsize

def resolution_bytes(rd):
    total = 0
    total += arr_bytes(rd.positions)
    total += arr_bytes(rd.velocities)
    total += arr_bytes(rd.potential)
    if getattr(rd, "potential_grid", None) is not None:
        total += arr_bytes(rd.potential_grid)
    if getattr(rd, "density_grid", None) is not None:
        total += arr_bytes(rd.density_grid)
    if getattr(rd, "grid", None) is not None:
        total += arr_bytes(rd.grid)
    return total

def print_batch_bytes(batch):
    lr = batch["lr"]; hr = batch["hr"]
    print("LR batch bytes:", lr.positions.shape, lr.positions.dtype, resolution_bytes(lr)/1024**3, "GiB")
    print("HR batch bytes:", hr.positions.shape, hr.positions.dtype, resolution_bytes(hr)/1024**3, "GiB")

print_batch_bytes(batch)

In [ ]:
os.environ["JAX_LOG_COMPILES"] = "1.0"
os.environ["JAX_TRACEBACK_FILTERING"] = "off"

In [ ]:
def info(name, x):
    print(f"{name}: type={type(x)} shape={getattr(x,'shape',None)} ndim={getattr(x,'ndim',None)}")

info("scale_factors", scale_factors)
info("max_idx", max_idx)

lr = batch["lr"]; hr = batch["hr"]
info("lr.positions", lr.positions)
info("lr.velocities", lr.velocities)
info("hr.positions", hr.positions)
info("hr.velocities", hr.velocities)

# si usas grid en loss
info("lr.potential_grid", lr.potential_grid)
info("lr.density_grid", lr.density_grid)
info("lr.grid", lr.grid)

In [ ]:
# ==========================================
# CELL 2: FORWARD PASS & GRADIENTES (MEM DEBUG)
# ==========================================

print_gpu_mem("before forward")

print_pytree_bytes("params", params)
print_pytree_bytes("batch", batch)  # ojo: si batch incluye simulación completa, aquí se ve enorme

# Ensure all arrays in batch are on the same device (GPU)
# from jax import device_put
# def move_batch_to_device(batch, device):
#     for res in ['lr', 'hr']:
#         for attr in ['grid', 'positions', 'velocities', 'potential', 'potential_grid', 'density_grid']:
#             if hasattr(batch[res], attr):
#                 arr = getattr(batch[res], attr)
#                 if hasattr(arr, 'device') and arr.device != device:
#                     setattr(batch[res], attr, device_put(arr, device))
#     return batch
print("scale_factors:", type(scale_factors), getattr(scale_factors, "shape", None), scale_factors)

# batch = move_batch_to_device(batch, jax.devices()[0])

# Forward + grad
train_loss, grads = jax.value_and_grad(train_loss_fn)(params, batch, scale_factors)
# Sync (muy importante)
train_loss.block_until_ready()

print_gpu_mem("after forward+grad")

print(f"Pérdida (Loss) del Batch: {float(train_loss):.5f}")

print_pytree_bytes("grads", grads)

# Opcional: inspeccionar el top-10 grad tensors más grandes
leaves = tree.tree_leaves(grads)
sizes = []
for i, x in enumerate(leaves):
    if hasattr(x, "shape") and hasattr(x, "dtype"):
        sizes.append((i, int(np.prod(x.shape) * np.dtype(x.dtype).itemsize), x.shape, x.dtype))
sizes.sort(key=lambda t: t[1], reverse=True)

print("\nTop 10 grad leaves by size:")
for i, nb, shape, dtype in sizes[:10]:
    print(f"  leaf {i:03d}: {pretty_bytes(nb)}  shape={shape} dtype={dtype}")

# Tu check de norms (con cuidado: linalg.norm en arrays grandes también aloca)
grad_norms = tree.tree_map(lambda x: jnp.linalg.norm(x).block_until_ready() if hasattr(x, "shape") else x, grads)

print("\nGrad norms (flattened):")
for norm in tree.tree_leaves(grad_norms)[:20]:
    print(float(norm))

In [ ]:
# ==========================================
# CELL 3: OPTIMIZER UPDATE (OPTILAX / OPTAX)
# ==========================================

# 1. Calculamos cómo deben cambiar los pesos basándonos en los gradientes
updates, opt_state = optimizer.update(grads, opt_state, params)

# 2. Aplicamos esos cambios a nuestra red neuronal
params = optax.apply_updates(params, updates)

print("¡Pesos actualizados exitosamente!")

# Si tu learning rate usa un schedule dinámico (cosine decay), podemos verlo así:
try:
    current_lr = opt_state.inner_opt_state[1].hyperparams["learning_rate"]
    print(f"Learning Rate actual: {current_lr}")
except AttributeError:
    print("Learning Rate es estático o la estructura del opt_state es diferente.")

In [ ]:
# ==========================================
# CELL 4: EVALUACIÓN / VALIDACIÓN MÚLTIPLE
# ==========================================

val_loss_total = 0.0

# Iteramos sobre todo el conjunto de validación
for val_batch in val_data:
    val_batch = val_data.move_to_device(val_batch, device=jax.devices()[0])
    
    # Nota: Aquí llamamos a loss_fn DIRECTAMENTE, sin value_and_grad
    # vl es el loss, val_pos_pm son las posiciones corregidas que devuelve tu función
    vl, val_pos_pm = loss_fn(params, val_batch, scale_factors)
    val_loss_total += vl

val_loss_mean = val_loss_total / len(val_data)
print(f"Validation Loss Media: {val_loss_mean:.5f}")

# Opcional: Probar tu función de graficado (asegúrate de que plt.show() funcione en el notebook)
plot_eval(val_pos_pm, val_batch,  fig_label="debug_val",use_wandb=False,plot_log=True,slab_thickness=128)

In [ ]:
from tqdm.notebook import tqdm

n_debug_steps = 100
pbar_debug = tqdm(range(n_debug_steps), desc="Entrenando mini-bucle (mem-debug)")

print_gpu_mem("start")
print_pytree_bytes("params (static)", params)

# TIP: fija max_idx para ver si el crecimiento es por recompilación
# fixed_max_idx = jnp.asarray(len(scale_factors)-1, dtype=jnp.int32)

for step in pbar_debug:
    # 1) max_idx
    if config.training.sample_snapshots:
        # rng, _ = jax.random.split(rng)
        # max_idx = jax.random.randint(
        #     rng, minval=10, maxval=len(scale_factors), shape=(1,), dtype=jnp.int32
        # )[0]
        max_idx = jnp.asarray(20, dtype=jnp.int32)
    else:
        max_idx = jnp.asarray(len(scale_factors)-1, dtype=jnp.int32)

    # 2) batch
    batch = next(train_data.iterator)

    # Diagnóstico: tamaño del batch en bytes (CPU o GPU)
    if step == 0:
        print_pytree_bytes("batch (before move)", batch)

    batch = train_data.move_to_device(batch, device=jax.devices()[0])

    if step == 0:
        # imprime shapes clave para confirmar si estás moviendo trayectoria completa
        print("LR positions:", batch["lr"].positions.shape, batch["lr"].positions.dtype,
              "device:", batch["lr"].positions.device)
        print("HR positions:", batch["hr"].positions.shape, batch["hr"].positions.dtype,
              "device:", batch["hr"].positions.device)
        if getattr(batch["lr"], "potential_grid", None) is not None:
            print("LR pot_grid:", batch["lr"].potential_grid.shape, batch["lr"].potential_grid.dtype,
                  "device:", batch["lr"].potential_grid.device)
        if getattr(batch["lr"], "density_grid", None) is not None:
            print("LR dens_grid:", batch["lr"].density_grid.shape, batch["lr"].density_grid.dtype,
                  "device:", batch["lr"].density_grid.device)

    # 3) forward+grad (medimos memoria antes y después)
    print_gpu_mem(f"step {step} before fwd")

    train_loss, grads = jax.value_and_grad(train_loss_fn)(params, batch, scale_factors)

    # 🔴 sincroniza para que la medición sea real
    train_loss.block_until_ready()
    print_gpu_mem(f"step {step} after fwd+grad")

    # 4) optim update (también puede alocar)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    # sincroniza una hoja de grads (opcional) para medir post-update
    # (bloquea usando un leaf pequeño o el loss ya sirve)
    train_loss.block_until_ready()
    print_gpu_mem(f"step {step} after update")

    pbar_debug.set_postfix({"Loss": f"{float(train_loss):.5f}", "max_idx": int(max_idx)})

    # 5) chequeo rápido de crecimiento sostenido
    if step == 0:
        print_pytree_bytes("grads (step0)", grads)

    # ✅ evita retener referencias: borra explícitamente
    del grads, updates, batch

    if step > 0 and step % 10 == 0:
        print(f"Step {step} | Loss: {float(train_loss):.5f} | max_idx={int(max_idx)}")

print_gpu_mem("end")
print("¡Mini-entrenamiento completado!")

In [ ]:
# ==========================================
# CELL 5: TEST SET & CHECKPOINTING
# ==========================================

# 1. Probar guardado local
# OJO: Asegúrate de que run_dir esté definido antes
print("Guardando checkpoint de prueba...")
# checkpoint(run_dir="/cosmos_storage/home/diegovillalba/JaxPM/models", loss=train_loss, params=params, prefix="debug_train", step=1)

# 2. Extraer un lote del Test Set
test_batch = val_data.move_to_device(test_data[0], device=jax.devices()[0])

# 3. Evaluar
test_loss, test_pos_pm = loss_fn(params, test_batch, scale_factors, max_idx=None)

print(f"Test Loss del modelo actual: {test_loss:.5f}")

In [ ]:
def train(config=None, data_dir=DEFAULT_DATA_DIR, output_dir=DEFAULT_MODEL_DIR):
    neural_net = build_network(config.correction_model)
    cosmology, scale_factors, train_data, val_data, test_data = build_dataloader(
        config.data, data_dir=data_dir
    )

    print(
        f"Dataset summary: Train ({len(train_data)}), Val ({len(val_data)}), Test ({len(test_data)})"
    )

    params = initialize_network(
        train_data[0], neural_net=neural_net, model_type=config.correction_model.type
    )

    # Setup WandB and directory
    run = wandb.init(
        project=config.wandb.project, config=config.to_dict(), dir=output_dir
    )
    print(f"Run name: {run.name}")
    run_dir = output_dir / f"{run.name}"
    run_dir.mkdir(exist_ok=True, parents=True)

    with open(run_dir / "config.yaml", "w") as f:
        yaml.dump(config.to_dict(), f)

    loss_fn = build_loss_fn(
        config.training,
        neural_net,
        cosmology,
        correction_type=config.correction_model.type,
        mesh_lr=train_data[0]["lr"].mesh,
    )

    schedule = build_schedule(config.training.schedule)
    optimizer, opt_state = build_optimizer(
        config.training, params=params, schedule=schedule
    )

    print_initial_lr_loss(val_data)

    early_stop = EarlyStopping(patience=config.training.patience)
    best_params = params
    best_loss = float("inf")
    rng = jax.random.PRNGKey(0)

    # Value and Grad wrapper for single step
    def train_loss_fn(p, batch, sf, midx):
        return loss_fn(params=p, dataset=batch, scale_factors=sf, max_idx=midx)[0]

    pbar = tqdm(range(config.training.n_steps), desc="Training")

    for step in pbar:
        # 1. Prepare Batch & Dynamic Slices
        if config.training.sample_snapshots:
            rng, _ = jax.random.split(rng)
            max_idx = jax.random.randint(
                rng, minval=10, maxval=len(scale_factors), shape=(1,)
            )[0]
        else:
            max_idx = None

        batch = next(train_data.iterator)
        batch = train_data.move_to_device(batch, device=jax.devices()[0])

        # 2. Gradient Update
        train_loss, grads = jax.value_and_grad(train_loss_fn)(
            params, batch, scale_factors, max_idx
        )
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)

        pbar.set_postfix({"Loss": float(train_loss)})

        # 3. Validation & Checkpointing
        eval_freq = 10 * config.training.batch_size
        if step > 0 and step % eval_freq == 0:
            val_loss = 0.0
            for val_batch in val_data:
                val_batch = val_data.move_to_device(val_batch, device=jax.devices()[0])
                vl, val_pos_pm = loss_fn(
                    params, val_batch, scale_factors, max_idx=max_idx
                )
                val_loss += vl

            val_loss /= len(val_data)

            # Plot validation sample
            plot_eval(val_pos_pm, val_batch, max_idx=max_idx)

            # Early stopping check
            early_stop = early_stop.update(val_loss)
            if early_stop.has_improved:
                best_params = params
                best_loss = val_loss

            # Step the schedule if it is a custom class that requires it
            if hasattr(schedule, "step"):
                schedule.step(val_loss)

            learning_rate = opt_state.inner_opt_state[1].hyperparams["learning_rate"]
            wandb.log(
                {
                    "train_loss": float(train_loss),
                    "val_loss": float(val_loss),
                    "learning_rate": float(learning_rate),
                },
                step=step,
            )

            pbar.set_postfix(
                {"train_loss": float(train_loss), "val_loss": float(val_loss)}
            )

            if early_stop.should_stop:
                print(f"Early stopping triggered at step {step}.")
                break

        # 4. Save Weights
        if step > 0 and step % config.training.checkpoint_every == 0:
            checkpoint(
                run_dir=run_dir,
                loss=train_loss,
                params=params,
                prefix="train",
                step=step,
            )

    # 5. Final Evaluation on Test Set
    checkpoint(run_dir=run_dir, params=best_params, loss=best_loss, prefix="best")

    test_batch = val_data.move_to_device(test_data[0], device=jax.devices()[0])
    test_loss, test_pos_pm = loss_fn(
        best_params, test_batch, scale_factors, max_idx=None
    )

    print(f"Test loss = {test_loss:.5f}")
    plot_eval(test_pos_pm, test_batch, max_idx=None, fig_label="test")

    wandb.finish()
    return best_loss